# Final Test Evaluation — Deep Models with Frozen Thresholds

Final test-set evaluation of the three deep models (TSM-fast, CNN-GRU, R(2+1)D-18) for the ES327 driver-drowsiness study. Loads the final v2 checkpoints, applies the **frozen decision thresholds** selected on validation, and evaluates on all three test sets.

**Locked thresholds (validation best-F1, Table 3.4):**

| Model | τ |
|---|---|
| TSM-fast | 0.010 |
| CNN-GRU | 0.001 |
| R(2+1)D-18 | 0.004 |

**Produces:** the final metrics table (Table 4.2), per-split precision-recall curves (Figures 4.3/4.4), and confusion matrices (Figure 4.1/4.2 and Appendix A.3).

---
## 1. Setup, Paths & Splits

In [ ]:
# ============================================================
# 0) Mount + paths
# ============================================================
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, pandas as pd
from collections import Counter

SPLITS_DIR = "/content/drive/MyDrive/ES327_Drowsiness/splits"
CLIPS_DRIVE = "/content/drive/MyDrive/ES327_Drowsiness/Clips"
CLIPS_LOCAL = "/content/Clips"

UTA_TEST   = os.path.join(SPLITS_DIR, "uta_test.csv")
DROZY_TEST = os.path.join(SPLITS_DIR, "drozy_test.csv")   # external robustness
YAWDD_TEST  = os.path.join(SPLITS_DIR, "yawdd_test.csv")   # add this

# ============================================================
# 1) Load split CSVs
# ============================================================
def load_split_csv(path):
    df = pd.read_csv(path)
    df = df.dropna(subset=["clip_path", "label"]).copy()
    df["label"] = df["label"].astype(int)
    return df

def namespace_subjects(df, default_ds):
    df = df.copy()
    if "dataset" in df.columns:
        ds = df["dataset"].fillna(default_ds).astype(str).str.lower()
    else:
        ds = pd.Series([default_ds]*len(df), index=df.index)
    df["subject_id"] = ds + "_" + df["subject_id"].astype(str)
    return df

uta_test   = namespace_subjects(load_split_csv(UTA_TEST),   "uta")
drozy_test = namespace_subjects(load_split_csv(DROZY_TEST), "drozy")
yawdd_test  = namespace_subjects(load_split_csv(YAWDD_TEST), "yawdd")  # add this


print("UTA_TEST:", len(uta_test), dict(Counter(uta_test["label"])))
print("DROZY_TEST:", len(drozy_test), dict(Counter(drozy_test["label"])))
print("YAWDD_TEST:", len(yawdd_test), dict(Counter(yawdd_test["label"])))  # add this


# ============================================================
# 2) Build rsync filelist for TEST + EXT
# ============================================================
all_paths = pd.concat([uta_test["clip_path"], drozy_test["clip_path"],yawdd_test["clip_path"]]).dropna().astype(str).unique()

prefix = CLIPS_DRIVE.rstrip("/") + "/"
rel_paths = [p[len(prefix):] for p in all_paths if p.startswith(prefix)]

print("TEST+EXT unique clips:", len(rel_paths))
print("Example:", rel_paths[0] if rel_paths else None)

filelist = "/content/needed_test_ext.txt"
with open(filelist, "w") as f:
    f.write("\n".join(rel_paths) + ("\n" if rel_paths else ""))

print("✅ Wrote:", filelist)

# ============================================================
# 3) Optional: remove known bad files by substring
#    (add more substrings if you ever find others)
# ============================================================
bad_subs = [
    "Fold3_part1__29__10__00122.npz",   # example from earlier
]

if bad_subs:
    with open(filelist, "r") as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    kept = []
    removed = 0
    for ln in lines:
        if any(bs in ln for bs in bad_subs):
            removed += 1
        else:
            kept.append(ln)

    with open(filelist, "w") as f:
        f.write("\n".join(kept) + ("\n" if kept else ""))

    print("List size before:", len(lines))
    print("List size after :", len(kept))
    print("Removed         :", removed)

# ============================================================
# 4) rsync Drive -> local SSD
# ============================================================


---
## 2. Cache Test Clips Locally

Copies the test-set clips to Colab's local SSD via rsync for fast loading. The batched helper (with progress bars) is convenient when copying large numbers of clips from Drive.

In [ ]:
!rsync -rlt --info=progress2 --partial --ignore-existing --ignore-errors \
  --files-from=/content/needed_uta_test.txt \
  "/content/drive/MyDrive/ES327_Drowsiness/Clips/" \
  "/content/Clips/"

In [ ]:
import os, math, time
import pandas as pd

# --- assumes you already created rel_paths (as in your code) ---
BATCH_SIZE = 2000

# Make sure local cache exists
os.makedirs(CLIPS_LOCAL, exist_ok=True)

# Convert to list once
rel_paths_list = list(rel_paths)
n = len(rel_paths_list)
n_batches = math.ceil(n / BATCH_SIZE)

print(f"Total files: {n} | Batch size: {BATCH_SIZE} | Batches: {n_batches}")

# Optional: keep a simple log
log_path = "/content/rsync_batches_log.txt"
with open(log_path, "w") as lf:
    lf.write(f"Total files: {n}\nBatch size: {BATCH_SIZE}\nBatches: {n_batches}\n\n")

for b in range(n_batches):
    start = b * BATCH_SIZE
    end = min((b + 1) * BATCH_SIZE, n)
    batch = rel_paths_list[start:end]

    batch_file = f"/content/needed_test_ext_batch_{b:03d}.txt"
    with open(batch_file, "w") as f:
        f.write("\n".join(batch) + "\n")

    print(f"\n=== Batch {b+1}/{n_batches} | files {start}:{end} ({end-start}) ===")
    t0 = time.time()

    # Run rsync for this batch
    cmd = f'''rsync -rlt \
      --info=progress2 \
      --partial \
      --ignore-existing \
      --ignore-errors \
      --files-from="{batch_file}" \
      "{CLIPS_DRIVE.rstrip('/')}/" \
      "{CLIPS_LOCAL.rstrip('/')}/"
    '''
    ret = os.system(cmd)

    dt = time.time() - t0
    with open(log_path, "a") as lf:
        lf.write(f"Batch {b+1}/{n_batches} | {start}:{end} | ret={ret} | {dt:.1f}s | file={batch_file}\n")

    print(f"Batch done. ret={ret} | time={dt/60:.1f} min")

print("\n✅ Finished batching. Log:", log_path)

In [ ]:
import os, math, time, subprocess
from tqdm import tqdm

BATCH_SIZE = 200
SLEEP_BETWEEN = 0

os.makedirs(CLIPS_LOCAL, exist_ok=True)

rel_paths_list = list(rel_paths)
n = len(rel_paths_list)
n_batches = math.ceil(n / BATCH_SIZE)

print(f"Total files: {n} | Batch size: {BATCH_SIZE} | Batches: {n_batches}")

def count_local():
    out = subprocess.check_output(
        ["bash","-lc","find /content/Clips -type f -name '*.npz' | wc -l"]
    )
    return int(out.decode().strip())

def run_rsync(batch_file):
    args = [
        "rsync", "-rlt",
        "--no-inc-recursive",
        "--partial",
        "--ignore-existing",
        "--ignore-errors",
        f"--files-from={batch_file}",
        f"{CLIPS_DRIVE.rstrip('/')}/",
        f"{CLIPS_LOCAL.rstrip('/')}/",
    ]
    return subprocess.run(args).returncode

with tqdm(total=n, desc="Total copied", unit="files") as pbar:
    already = count_local()
    pbar.update(already)

    for b in range(n_batches):
        start = b * BATCH_SIZE
        end = min((b + 1) * BATCH_SIZE, n)
        batch = rel_paths_list[start:end]

        batch_file = f"/content/batch_{b:03d}.txt"
        with open(batch_file, "w") as f:
            f.write("\n".join(batch) + "\n")

        ret = run_rsync(batch_file)

        now = count_local()
        pbar.update(now - already)
        already = now

        time.sleep(SLEEP_BETWEEN)

print("✅ Done.")

---
## 3. Model Builders

Defines the three architectures (3D-ResNet-18, CNN-GRU, TSM-fast).

In [ ]:
import torchvision
import torch.nn as nn

NUM_CLASSES = 2

# 3D-CNN: 3D ResNet-18
def build_r3d18(num_classes=2):
    m = torchvision.models.video.r3d_18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

# CNN-GRU: ResNet18 per-frame + GRU
class ResNetGRU(nn.Module):
    def __init__(self, hidden=256, num_classes=2):
        super().__init__()
        base = torchvision.models.resnet18(weights=None)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.feat_dim = 512
        self.gru = nn.GRU(self.feat_dim, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, num_classes)

    def forward(self, x):  # (B,T,C,H,W)
        B,T,C,H,W = x.shape
        x = x.reshape(B*T, C, H, W)
        f = self.backbone(x).flatten(1)
        f = f.reshape(B, T, self.feat_dim)
        out, _ = self.gru(f)
        return self.fc(out[:, -1, :])

def build_cnn_gru(num_classes=2):
    return ResNetGRU(hidden=128, num_classes=num_classes)

# TSM-style temporal average baseline
class ResNetTemporalAvg(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        base = torchvision.models.resnet18(weights=None)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):  # (B,C,T,H,W)
        B,C,T,H,W = x.shape
        x = x.permute(0,2,1,3,4).contiguous().view(B*T, C, H, W)
        f = self.backbone(x).flatten(1).view(B, T, 512).mean(dim=1)
        return self.fc(f)

def build_tsm_fast(num_classes=2):
    return ResNetTemporalAvg(num_classes=num_classes)

print("✅ Builders defined.")

---
## 4. Final Evaluation (Frozen Thresholds)

Loads the v2 `best.pt` checkpoints, applies the locked thresholds (τ = 0.010 / 0.001 / 0.004), and evaluates TSM-fast, CNN-GRU and R(2+1)D-18 on the UTA-RLDD, YawDD and DROZY test sets. Saves `final_test_metrics.csv` and per-clip predictions.

In [ ]:
# ============================================================
# 0) Imports + paths
# ============================================================
import os, numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, confusion_matrix
)
import zipfile
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

SPLITS_DIR   = "/content/drive/MyDrive/ES327_Drowsiness/splits"
CLIPS_DRIVE  = "/content/drive/MyDrive/ES327_Drowsiness/Clips"
CLIPS_LOCAL  = "/content/Clips"

OUT_DIR = "/content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models_v2/final_test_eval"
os.makedirs(OUT_DIR, exist_ok=True)

# Split CSVs
UTA_TEST    = os.path.join(SPLITS_DIR, "uta_test.csv")
DROZY_TEST  = os.path.join(SPLITS_DIR, "drozy_test.csv")
YAWDD_TEST  = os.path.join(SPLITS_DIR, "yawdd_test.csv")

# Checkpoints
CKPT_TSM   = "/content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models_v2/tsm_fast/best.pt"
CKPT_GRU   = "/content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models_v2/cnn_gru/best.pt"
CKPT_R3D18 = "/content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models_v2/r3d18/best.pt"

# Locked thresholds (VAL-bestF1)
THR = {"tsm_fast": 0.010, "cnn_gru": 0.001, "r3d18": 0.004}

# ============================================================
# 1) CSV helpers
# ============================================================
def load_split_csv(path):
    df = pd.read_csv(path)
    df = df.dropna(subset=["clip_path", "label"]).copy()
    df["label"] = df["label"].astype(int)
    return df

def namespace_subjects(df, default_ds):
    df = df.copy()
    if "dataset" in df.columns:
        ds = df["dataset"].fillna(default_ds).astype(str).str.lower()
    else:
        ds = pd.Series([default_ds]*len(df), index=df.index)
    df["subject_id"] = ds + "_" + df["subject_id"].astype(str)
    return df

# ============================================================
# 2) Dataset (skips corrupt npz and logs bad paths)
# ============================================================
class NPZClipsDataset(Dataset):
    def __init__(self, df, clips_drive, clips_local, mode="cthw", bad_log_path=None):
        self.df = df.reset_index(drop=True)
        self.clips_drive = clips_drive.rstrip("/")
        self.clips_local = clips_local.rstrip("/")
        self.mode = mode
        self.bad_log_path = bad_log_path

    def _map_to_local(self, p):
        p = str(p)
        if p.startswith(self.clips_drive + "/"):
            rel = p[len(self.clips_drive)+1:]
            local = os.path.join(self.clips_local, rel)
            return local, p
        return None, p

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        drive_path = row["clip_path"]
        y = int(row["label"])

        local_path, original = self._map_to_local(drive_path)
        path_to_use = local_path if (local_path and os.path.exists(local_path)) else original

        try:
            data = np.load(path_to_use)
            x = data["clip"] if "clip" in data else data["arr_0"]
        except (zipfile.BadZipFile, OSError, ValueError) as e:
            if self.bad_log_path:
                with open(self.bad_log_path, "a") as f:
                    f.write(f"{path_to_use}\n")
            return None

        x = x.astype(np.float32)
        if x.max() > 1.5:
            x = x / 255.0

        # dataset stored (C,T,H,W)
        if self.mode == "tchw":
            x = np.transpose(x, (1,0,2,3))  # (T,C,H,W)

        return torch.from_numpy(x), torch.tensor(y, dtype=torch.long), str(path_to_use)

def collate_drop_none(batch):
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None
    xs, ys, ps = zip(*batch)
    return torch.stack(xs, 0), torch.stack(ys, 0), list(ps)

# ============================================================
# 3) Model loading (handles checkpoint bundles)
# ============================================================
def load_model(builder_fn, ckpt_path):
    model = builder_fn()
    ckpt = torch.load(ckpt_path, map_location="cpu")

    if isinstance(ckpt, dict):
        if "model_state" in ckpt:
            sd = ckpt["model_state"]
        elif "state_dict" in ckpt:
            sd = ckpt["state_dict"]
        elif "model" in ckpt and isinstance(ckpt["model"], dict):
            sd = ckpt["model"]
        else:
            sd = ckpt
    else:
        sd = ckpt

    model.load_state_dict(sd, strict=True)
    model.to(device).eval()
    return model

# ============================================================
# 4) Inference + metrics
# ============================================================
@torch.no_grad()
def predict_probs(model, loader, model_name, split_name):
    probs, ys, paths = [], [], []
    dropped_batches = 0

    for batch in tqdm(loader, desc=f"{split_name} | {model_name}", leave=False):
        if batch is None:
            dropped_batches += 1
            continue

        xb, yb, pb = batch
        xb = xb.to(device, non_blocking=True)
        yb = yb.cpu().numpy().tolist()

        out = model(xb)

        if out.ndim == 1:
            p = torch.sigmoid(out)
        elif out.ndim == 2 and out.shape[1] == 1:
            p = torch.sigmoid(out[:, 0])
        elif out.ndim == 2 and out.shape[1] == 2:
            p = torch.softmax(out, dim=1)[:, 1]
        else:
            raise ValueError(f"Unexpected output shape: {tuple(out.shape)}")

        probs.extend(p.detach().cpu().numpy().tolist())
        ys.extend(yb)
        paths.extend(list(pb))

    return np.array(probs), np.array(ys), np.array(paths), dropped_batches

def compute_metrics(y_true, p, thr):
    y_pred = (p >= thr).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp + 1e-12)

    return {
        "threshold": float(thr),
        "F1": float(f1_score(y_true, y_pred)),
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "Recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "Specificity": float(specificity),
        "PR_AUC": float(average_precision_score(y_true, p)),
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
    }

def run_eval(split_name, df, model_key, model, mode, batch_size=64, num_workers=0):
    bad_log = os.path.join(OUT_DIR, f"bad_npz_files_{split_name}.txt")
    # clear previous log for this run
    if os.path.exists(bad_log):
        os.remove(bad_log)

    ds = NPZClipsDataset(
        df, clips_drive=CLIPS_DRIVE, clips_local=CLIPS_LOCAL,
        mode=mode, bad_log_path=bad_log
    )
    loader = DataLoader(
        ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True,
        collate_fn=collate_drop_none
    )

    p, y, paths, dropped_batches = predict_probs(model, loader, model_key, split_name)

    # Count bad files logged
    bad_count = 0
    if os.path.exists(bad_log):
        with open(bad_log, "r") as f:
            bad_count = sum(1 for _ in f)

    thr = THR[model_key]
    m = compute_metrics(y, p, thr)

    # Save per-clip predictions
    pred_df = pd.DataFrame({"path": paths, "y_true": y, "p_drowsy": p})
    pred_path = os.path.join(OUT_DIR, f"preds_{model_key}_{split_name}.csv")
    pred_df.to_csv(pred_path, index=False)

    m_row = {
        "split": split_name,
        "model": model_key,
        "n_total_csv": int(len(df)),
        "n_used": int(len(y)),
        "n_bad_npz": int(bad_count),
        "dropped_empty_batches": int(dropped_batches),
        **m
    }

    print(split_name, model_key, "->",
          {k: m_row[k] for k in ["n_used","n_bad_npz","F1","PR_AUC","Recall","Precision","Accuracy","threshold"]})

    return m_row, pred_path

# ============================================================
# 5) Load splits
# ============================================================
uta_test   = namespace_subjects(load_split_csv(UTA_TEST),   "uta")
drozy_test = namespace_subjects(load_split_csv(DROZY_TEST), "drozy")
yawdd_test = namespace_subjects(load_split_csv(YAWDD_TEST), "yawdd")

print("UTA_TEST:", len(uta_test))
print("YAWDD_TEST:", len(yawdd_test))
print("DROZY_TEST:", len(drozy_test))

# ============================================================
# 6) Load models (builders must already be defined)
# ============================================================
tsm   = load_model(build_tsm_fast, CKPT_TSM)
gru   = load_model(build_cnn_gru,  CKPT_GRU)
r3d18 = load_model(build_r3d18,    CKPT_R3D18)
print("✅ Loaded all models")

# ============================================================
# 7) Run final evaluation
# ============================================================
rows = []

# TSM-fast: cthw
for split_name, df in [("UTA_test", uta_test), ("YawDD_test", yawdd_test), ("DROZY_test", drozy_test)]:
    r, _ = run_eval(split_name, df, "tsm_fast", tsm, mode="cthw", batch_size=64, num_workers=0)
    rows.append(r)

# CNN-GRU: tchw
for split_name, df in [("UTA_test", uta_test), ("YawDD_test", yawdd_test), ("DROZY_test", drozy_test)]:
    r, _ = run_eval(split_name, df, "cnn_gru", gru, mode="tchw", batch_size=64, num_workers=0)
    rows.append(r)

# R3D18: cthw (heavier)
for split_name, df in [("UTA_test", uta_test), ("YawDD_test", yawdd_test), ("DROZY_test", drozy_test)]:
    r, _ = run_eval(split_name, df, "r3d18", r3d18, mode="cthw", batch_size=32, num_workers=0)
    rows.append(r)

final_df = pd.DataFrame(rows)
final_path = os.path.join(OUT_DIR, "final_test_metrics.csv")
final_df.to_csv(final_path, index=False)

print("\n✅ Saved:", final_path)
final_df

---
## 5. Results Table (Table 4.2)

Formats the final metrics into the dissertation results table (per-dataset F1, PR-AUC, accuracy, precision, recall, specificity), ordered by dataset and model.

In [ ]:
import os, pandas as pd, numpy as np
import matplotlib.pyplot as plt

OUT_DIR = "/content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models_v2/final_test_eval"
metrics_path = os.path.join(OUT_DIR, "final_test_metrics.csv")

df = pd.read_csv(metrics_path)

# Nice names + ordering
split_order = ["UTA_test", "YawDD_test", "DROZY_test"]
model_order = ["tsm_fast", "cnn_gru", "r3d18"]
model_name = {"tsm_fast":"TSM-fast", "cnn_gru":"CNN-GRU", "r3d18":"R3D18"}

df["split"] = pd.Categorical(df["split"], categories=split_order, ordered=True)
df["model"] = pd.Categorical(df["model"], categories=model_order, ordered=True)
df = df.sort_values(["split","model"]).copy()

# Format columns you likely want in the dissertation
tbl = df[[
    "split","model","threshold","F1","PR_AUC","Accuracy","Precision","Recall","Specificity",
    "n_used","n_bad_npz"
]].copy()

tbl["model"] = tbl["model"].map(model_name)
tbl["threshold"] = tbl["threshold"].map(lambda x: f"{x:.3f}")
for c in ["F1","PR_AUC","Accuracy","Precision","Recall","Specificity"]:
    tbl[c] = tbl[c].map(lambda x: f"{x:.3f}")

# Save CSV
csv_out = os.path.join(OUT_DIR, "final_test_results_table.csv")
tbl.to_csv(csv_out, index=False)
print("✅ Saved CSV:", csv_out)

# Save PNG (matplotlib table)
png_out = os.path.join(OUT_DIR, "final_test_results_table.png")

fig_w = 16
fig_h = 0.5 + 0.35*len(tbl)  # scale with rows
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.axis("off")

table = ax.table(
    cellText=tbl.values,
    colLabels=tbl.columns.tolist(),
    loc="center",
    cellLoc="center"
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.4)

plt.tight_layout()
plt.savefig(png_out, dpi=250, bbox_inches="tight")
plt.close(fig)

print("✅ Saved PNG:", png_out)

---
## 6. Precision-Recall Curves

Per-split PR curves for all three models, showing ranking performance independent of the chosen threshold (Figures 4.3/4.4, Appendix A.1).

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score

OUT_DIR = "/content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models_v2/final_test_eval"

splits = ["UTA_test", "YawDD_test", "DROZY_test"]
models = ["tsm_fast", "cnn_gru", "r3d18"]
model_name = {"tsm_fast":"TSM-fast", "cnn_gru":"CNN-GRU", "r3d18":"R3D18"}

for split in splits:
    plt.figure(figsize=(7,5))

    for m in models:
        pred_path = os.path.join(OUT_DIR, f"preds_{m}_{split}.csv")
        pred = pd.read_csv(pred_path)

        y = pred["y_true"].astype(int).values
        p = pred["p_drowsy"].astype(float).values

        prec, rec, _ = precision_recall_curve(y, p)
        ap = average_precision_score(y, p)

        # Do NOT set colors (per your environment rules)
        plt.plot(rec, prec, label=f"{model_name[m]} (AP={ap:.3f})")

    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Precision–Recall Curve: {split}")
    plt.legend()
    plt.grid(True, alpha=0.3)

    out_png = os.path.join(OUT_DIR, f"pr_curve_{split}.png")
    plt.tight_layout()
    plt.savefig(out_png, dpi=250)
    plt.close()

    print("✅ Saved:", out_png)

---
## 7. Confusion Matrices

Confusion matrices for each model on each test set at the frozen thresholds, illustrating the recall/specificity trade-offs discussed in the analysis (Figure 4.1/4.2, Appendix A.3).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

OUT_DIR = "/content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models_v2/final_test_eval"

splits = ["UTA_test", "YawDD_test", "DROZY_test"]
models = ["tsm_fast", "cnn_gru", "r3d18"]
model_name = {"tsm_fast":"TSM-fast", "cnn_gru":"CNN-GRU", "r3d18":"R3D18"}

# Locked thresholds
THR = {"tsm_fast": 0.010, "cnn_gru": 0.001, "r3d18": 0.004}

def plot_cm(cm, title, out_path):
    # cm in order [[TN, FP],[FN, TP]]
    fig, ax = plt.subplots(figsize=(5.2, 4.6))
    ax.imshow(cm)  # no manual colors

    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["0 (Alert)", "1 (Drowsy)"])
    ax.set_yticklabels(["0 (Alert)", "1 (Drowsy)"])

    # annotate
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, str(v), ha="center", va="center")

    plt.tight_layout()
    plt.savefig(out_path, dpi=250, bbox_inches="tight")
    plt.close(fig)

for split in splits:
    for m in models:
        pred_path = os.path.join(OUT_DIR, f"preds_{m}_{split}.csv")
        pred = pd.read_csv(pred_path)

        y = pred["y_true"].astype(int).values
        p = pred["p_drowsy"].astype(float).values

        thr = THR[m]
        yhat = (p >= thr).astype(int)

        cm = confusion_matrix(y, yhat, labels=[0,1])  # [[TN,FP],[FN,TP]]

        title = f"Confusion Matrix: {model_name[m]} on {split} (thr={thr})"
        out_png = os.path.join(OUT_DIR, f"cm_{m}_{split}.png")

        plot_cm(cm, title, out_png)
        print("✅ Saved:", out_png)